# Start here

This notebook checks that your FinUties key works against the live API at `https://data.finuties.com`.

## What
A four-step smoke test: load `FINUTIES_API_KEY`, hit `/health` and `/api/v1/status/p0`, list public recipes, then run one small COT query.

## Why this model
Before any chart, confirm auth, API reachability, and that a documented public endpoint returns rows. That is the repeatable baseline.

## How to rerun
1. Copy `notebooks/.env.example` to `notebooks/.env`.
2. Set `FINUTIES_API_KEY` from `POST https://data.finuties.com/api/v1/auth/sandbox` (the JSON `key` field).
3. Run every cell top to bottom.

Sandbox keys expire in about 72 hours. Longer-lived keys live in [FinUties settings](https://www.finuties.com/settings).

In [1]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv


def resolve_notebooks_env(start_dir: Path) -> Path:
    current = start_dir.resolve()
    for candidate_root in [current, *current.parents]:
        if candidate_root.name == "notebooks":
            env_path = candidate_root / ".env"
            if env_path.exists():
                return env_path
        nested_env = candidate_root / "notebooks" / ".env"
        if nested_env.exists():
            return nested_env
    raise FileNotFoundError(
        "Missing notebooks/.env. Copy notebooks/.env.example and set FINUTIES_API_KEY. "
        "A sandbox key is POST https://data.finuties.com/api/v1/auth/sandbox"
    )


def require_frame(df: pd.DataFrame, required: list[str], min_rows: int = 1) -> None:
    if df.empty:
        raise AssertionError("Expected a non-empty frame from the FinUties API.")
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise AssertionError(f"Missing required columns {missing}. Got {list(df.columns)}")
    if len(df) < min_rows:
        raise AssertionError(f"Expected at least {min_rows} rows, got {len(df)}")


def require_finite(series: pd.Series, name: str) -> None:
    numeric = pd.to_numeric(series, errors="coerce")
    valid = numeric.dropna()
    if valid.empty:
        raise AssertionError(f"{name} has no numeric values")
    if not np.isfinite(valid.to_numpy()).all():
        raise AssertionError(f"{name} contains non-finite values")


load_dotenv(resolve_notebooks_env(Path.cwd()))
API_ORIGIN = os.getenv("FINUTIES_API_ORIGIN", "https://data.finuties.com").rstrip("/")
API_KEY = os.getenv("FINUTIES_API_KEY", "").strip()
if not API_KEY:
    raise ValueError(
        "Missing FINUTIES_API_KEY. Copy notebooks/.env.example to notebooks/.env "
        "and set a key from POST /api/v1/auth/sandbox"
    )

HEADERS = {"Authorization": f"Bearer {API_KEY}"}
TIMEOUT_SECONDS = 45


def finuties_get(endpoint: str, params: dict | None = None):
    response = requests.get(
        f"{API_ORIGIN}{endpoint}",
        headers=HEADERS,
        params=params or {},
        timeout=TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    return response.json()


def normalize_rows(payload) -> list[dict]:
    if isinstance(payload, list):
        return [row for row in payload if isinstance(row, dict)]
    if isinstance(payload, dict):
        for key in ("items", "data", "rows", "results"):
            rows = payload.get(key)
            if isinstance(rows, list):
                return [row for row in rows if isinstance(row, dict)]
    return []


## Health and P0 status

`/health` is unauthenticated. `/api/v1/status/p0` is the public coverage surface (filings, ownership, equities, positioning, rates, economics, calendar, system). `overall` can be green, yellow, or red — red means some domain is stale, not that the API is down.

In [2]:
health = requests.get(f"{API_ORIGIN}/health", timeout=TIMEOUT_SECONDS)
health.raise_for_status()
health_payload = health.json()
print("health:", health_payload)
assert health_payload.get("status") == "online", health_payload

p0 = finuties_get("/api/v1/status/p0")
require_frame(pd.DataFrame(p0.get("domains") or []), ["domain", "status"], min_rows=1)
print("p0 overall:", p0.get("overall"))
print("p0 summary:", p0.get("summary"))
pd.DataFrame(p0["domains"])[["domain", "status", "budget_hours"]]

health: {'status': 'online'}


p0 overall: red
p0 summary: {'green': 1, 'yellow': 4, 'red': 2, 'total': 8}


,domain,status,budget_hours
0,calendar,unknown,12
1,economics,red,48
2,equities,yellow,6
3,filings,green,24
4,ownership,yellow,48
5,positioning,yellow,48
6,rates,red,24
7,system,yellow,6


## Recipes, then one query

`GET /api/v1/recipes` is the public job list. The tiny query is wheat COT facts (`GET /api/v1/cftc/legacy_futures-facts`) — verified live, not a guessed `/api/v1/data/...` path.

In [3]:
recipes_payload = finuties_get("/api/v1/recipes")
recipes = pd.DataFrame(recipes_payload.get("recipes") or [])
require_frame(recipes, ["id", "title", "path"], min_rows=1)
print(f"recipes: {len(recipes)}")

cot_payload = finuties_get(
    "/api/v1/cftc/legacy_futures-facts",
    {"commodity": "WHEAT", "limit": 5},
)
cot = pd.DataFrame(normalize_rows(cot_payload))
require_frame(cot, ["commodity_name", "report_date_as_yyyy_mm_dd"], min_rows=1)
assert (cot["commodity_name"] == "WHEAT").any(), "Expected WHEAT rows from the COT query"
print(f"COT sample rows: {len(cot)}")
recipes[["id", "title", "path"]]

recipes: 4


COT sample rows: 5


,id,title,path
0,13f-qoq,13F quarter-over-quarter holder changes,/api/v1/holdings/stock/{symbol}/changes
1,calendar-tonight,Tonight's economic calendar,/api/v1/query
2,fed-treasury-panel,Fed / Treasury rates panel,/api/v1/data/rates/nyfed-reference
3,dow30-sandbox-smoke,Dow-30 style sandbox smoke,/api/v1/status/p0


## Caveats

- A 200 from `/health` does not mean every dataset is fresh. Read `status/p0`.
- Sandbox keys are short-lived and rate-limited.
- Do not commit `notebooks/.env`.

Next: `money_flow/latest_cot_data_example.ipynb`, then commodities, macro, equity flows, and risk models.